# สกัดข้อมูลพิกัดคู่แข่งธุรกิจอาหารในกรุงเทพฯ (Competitors Extraction)
สมุดโน้ตเล่มนี้ถูกปรับปรุงให้ดึงข้อมูลร้านอาหาร, ร้านกาแฟ, ร้านฟาสต์ฟู้ด และบาร์ ผ่าน **Overpass API** (ดึงผ่านอินเทอร์เน็ตโดยตรง) ทำให้ไม่ต้องแกะไฟล์แผนที่ดิบ `osm.pbf` ขนาด 324MB ในเครื่อง ซึ่งจะช่วยย่นระยะเวลาการประมวลผลให้รวดเร็วและประหยัดทรัพยากรแรม

In [ ]:
import pandas as pd
import requests
import os
import gc
from pathlib import Path
from folium.plugins import MarkerCluster
import folium

# ค้นหาตำแหน่งโฟลเดอร์หลักของโปรเจกต์
NOTEBOOK_DIR = Path(os.getcwd())
BASE_DIR = NOTEBOOK_DIR.parent.parent # เลื่อนขึ้นไปที่ ZoneVision/data-pipeline

print(f"Notebook Directory: {NOTEBOOK_DIR}")
print(f"Base Directory: {BASE_DIR}")

In [ ]:
# โหลดตั้งค่าขอบเขตรอยต่อกรุงเทพฯ (BBox) จากคอนฟิก
import json
config_path = BASE_DIR / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

bkk_bbox = config.get("bkk_bbox", [100.30, 13.45, 100.95, 13.95])
target_competitors = config.get("target_competitors", ['restaurant', 'cafe', 'fast_food', 'bar'])

# Bbox format for Overpass API
bbox_str = f"{bkk_bbox[1]},{bkk_bbox[0]},{bkk_bbox[3]},{bkk_bbox[2]}"
print(f"Bangkok BBox: {bbox_str}")
print(f"Target Competitors: {target_competitors}")

In [ ]:
# 1. ดึงข้อมูลจาก Overpass API
print("กำลังดาวน์โหลดข้อมูลกลุ่มคู่แข่งจาก Overpass API...")
overpass_url = "http://overpass-api.de/api/interpreter"
amenity_regex = "|".join(target_competitors)

query_str = f"""
[out:json][timeout:120];
(
  node["amenity"~"{amenity_regex}"]({bbox_str});
  way["amenity"~"{amenity_regex}"]({bbox_str});
);
out center;
"""

headers = {
    'User-Agent': 'ZoneVisionSeniorProject/1.0 (contact: naeiger@example.com)'
}

try:
    response = requests.post(overpass_url, data={'data': query_str}, headers=headers, timeout=120)
    if response.status_code == 200:
        data = response.json()
        elements = data.get('elements', [])
        print(f"🎉 ดึงข้อมูลสำเร็จ! พบรายการข้อมูลคู่แข่ง: {len(elements)} รายการ")
        
        competitors = []
        for el in elements:
            lat = el.get('lat') or el.get('center', {}).get('lat')
            lon = el.get('lon') or el.get('center', {}).get('lon')
            tags = el.get('tags', {})
            name = tags.get('name') or tags.get('name:en') or tags.get('name:th')
            amenity = tags.get('amenity')
            cuisine = tags.get('cuisine')
            
            competitors.append({
                'name': name,
                'amenity_type': amenity,
                'cuisine': cuisine,
                'latitude': lat,
                'longitude': lon
            })
            
        df_pois_flat = pd.DataFrame(competitors)
        print(df_pois_flat.head(10))
    else:
        print(f"❌ การดึงข้อมูลล้มเหลว: HTTP Code {response.status_code}")
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดึงข้อมูล: {str(e)}")

In [ ]:
# 2. บันทึกผลลัพธ์ข้อมูลระดับดิบลงในคลัง interim
if 'df_pois_flat' in locals() and len(df_pois_flat) > 0:
    os.makedirs(str(BASE_DIR / "data" / "interim"), exist_ok=True)
    output_competitor_file = str(BASE_DIR / "data" / "interim" / "bangkok_competitors.json")
    df_pois_flat.to_json(output_competitor_file, orient='records', force_ascii=False, indent=4)
    print(f"💾 บันทึกไฟล์ระดับดิบเข้าคลัง interim สำเร็จ: {output_competitor_file}")

In [ ]:
# 3. วาดแผนที่แสดงผลคู่แข่งด้วย Folium
if 'df_pois_flat' in locals() and len(df_pois_flat) > 0:
    print("🗺️ กำลังประมวลผลวาดแผนที่คู่แข่ง... (ดึง 2,000 ร้านแรกมาพล็อตเพื่อความเร็ว)")
    m_comp = folium.Map(location=[13.7563, 100.5018], zoom_start=11)
    marker_cluster_comp = MarkerCluster().add_to(m_comp)

    comp_color_map = {
        'restaurant': 'orange',
        'cafe': 'purple',
        'fast_food': 'red',
        'bar': 'black'
    }

    for idx, row in df_pois_flat.head(2000).iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"<b>{row['name']}</b><br>ประเภท: {row['amenity_type']}<br>สไตล์อาหาร: {row['cuisine']}",
            icon=folium.Icon(color=comp_color_map.get(row['amenity_type'], 'gray'), icon='shopping-cart')
        ).add_to(marker_cluster_comp)

    map_output_path = str(BASE_DIR / "data" / "processed" / "bangkok_competitors_map.html")
    m_comp.save(map_output_path)
    print(f"💾 🎉 บันทึกแผนที่ HTML สำเร็จ! จัดเก็บไว้ที่: {map_output_path}")
    
    # เคลียร์แรมระบบข้อมูล
    del m_comp
    del marker_cluster_comp
    gc.collect()